In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel,AutoModelForSequenceClassification,DataCollatorWithPadding,TrainingArguments,Trainer
from datasets import Dataset
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
import ast
ds=pd.read_csv("out.csv")
ds["tag"]=ds["tag"].apply(ast.literal_eval)
ds

,thread_id,text,tag
0,2,"Let $ABC$ be a triangle, and $M$ an interior p...",[geometry]
1,3,okay this one is from Prof. Mircea Lascu from ...,"[algebra, geometry]"
2,5,"If A,B are invertible and the set {Ak - Bk | k...",[algebra]
3,9,In a magic square $n \times n$ composed from t...,"[algebra, combinatorics]"
4,72,The lengths of the sides of a convex hexagon $...,[geometry]
...,...,...,...
46451,36238654,"Let $a, b, c$ be the altitudes of triangle $A$...",[geometry]
46452,36238689,Find all functions that satisfy the condition ...,[algebra]
46453,36238706,On an $N \times N$ “chessboard” ($N \ge 3$) ea...,[combinatorics]
46454,36238733,It is known that $(20 + 25)^2 = 2025$. Find al...,[number theory]


In [3]:
tokenizer=AutoTokenizer.from_pretrained("tbs17/MathBERT")
# model=AutoModelForSequenceClassification.from_pretrained("bert-base-cased",num_labels=10)

In [4]:
type(ds["tag"][0])

list

In [5]:
mlb=MultiLabelBinarizer()
Y=mlb.fit_transform(ds["tag"]).astype(float)
print(mlb.classes_)

['algebra' 'combinatorics' 'geometry' 'number theory']


In [6]:
Y

array([[0., 0., 1., 0.],
       [1., 0., 1., 0.],
       [1., 0., 0., 0.],
       ...,
       [0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 1., 0.]], shape=(46456, 4))

In [7]:
dataset=Dataset.from_dict({"text":ds["text"],"labels":Y})


In [8]:
dataset

Dataset({
    features: ['text', 'labels'],
    num_rows: 46456
})

In [9]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/46456 [00:00<?, ? examples/s]

In [10]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset["train"]
temp_dataset = dataset["test"]
temp_dataset = temp_dataset.train_test_split(
    test_size=0.5,
    seed=42
)

eval_dataset = temp_dataset["train"]
test_dataset = temp_dataset["test"]

In [11]:
train_dataset[0]

{'text': 'Let $ABC$ be an obtuse triangle with $AB = AC$, and let $\\Gamma$ be the circle that is tangent to $AB$ at $B$ and to $AC$ at $C$. Let $D$ be the point on $\\Gamma$ furthest from $A$ such that $AD$ is perpendicular to $BC$. Point $E$ is the intersection of $AB$ and $DC$, and point $F$ lies on line $AB$ such that $BC = BF$ and $B$ lies on segment $AF$. Finally, let $P$ be the intersection of lines $AC$ and $DB$. Show that $PE = PF$.',
 'labels': [0.0, 0.0, 1.0, 0.0],
 'input_ids': [101,
  2292,
  1002,
  5925,
  1002,
  2022,
  2019,
  27885,
  5809,
  2063,
  9546,
  2007,
  1002,
  11113,
  1027,
  9353,
  1002,
  1010,
  1998,
  2292,
  1002,
  1032,
  13091,
  1002,
  2022,
  1996,
  4418,
  2008,
  2003,
  27250,
  2000,
  1002,
  11113,
  1002,
  2012,
  1002,
  1038,
  1002,
  1998,
  2000,
  1002,
  9353,
  1002,
  2012,
  1002,
  1039,
  1002,
  1012,
  2292,
  1002,
  1040,
  1002,
  2022,
  1996,
  2391,
  2006,
  1002,
  1032,
  13091,
  1002,
  6519,
  20515,
  20

In [12]:
test_dataset

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4646
})

In [13]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [14]:
model=AutoModelForSequenceClassification.from_pretrained("tbs17/MathBERT",num_labels=4,problem_type="multi_label_classification")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: tbs17/MathBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

In [15]:

training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",      # nếu bạn dùng transformers cũ thì đổi thành evaluation_strategy="epoch"
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=10,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="jaccard",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none" # tắt wandb cho đỡ lỗi
)

In [ ]:
import numpy as np
from sklearn.metrics import (
    jaccard_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred


    probs = 1 / (1 + np.exp(-logits))


    preds = (probs >= 0.5).astype(int)
    labels = labels.astype(int)


    label_acc = np.mean(preds == labels, axis=0)

    jaccard_samples = jaccard_score(
        labels,
        preds,
        average="samples",
        zero_division=0
    )


    f1_micro = f1_score(
        labels, preds,
        average="micro",
        zero_division=0
    )

    f1_macro = f1_score(
        labels, preds,
        average="macro",
        zero_division=0
    )

    precision_micro = precision_score(
        labels, preds,
        average="micro",
        zero_division=0
    )

    recall_micro = recall_score(
        labels, preds,
        average="micro",
        zero_division=0
    )

    return {
        **{
            f"acc_label_{i}": float(acc)
            for i, acc in enumerate(label_acc)
        },

        "jaccard_samples": float(jaccard_samples),


        "f1_micro": float(f1_micro),
        "f1_macro": float(f1_macro),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
    }


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

d:\Dev\Hoang\AI\MathClassification\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\huyho\.cache\huggingface\hub\models--tbs17--MathBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Jaccard
1,0.201144,0.175147,0.862462
2,0.164919,0.168357,0.872830
3,0.105998,0.178692,0.877726


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6969, training_loss=0.14205227097987377, metrics={'train_runtime': 4474.6294, 'train_samples_per_second': 24.916, 'train_steps_per_second': 1.557, 'total_flos': 1.3056191459665536e+16, 'train_loss': 0.14205227097987377, 'epoch': 3.0})

In [20]:
model=AutoModelForSequenceClassification.from_pretrained("../MathBERT_finetune/checkpoint-4646")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [22]:
trainer.evaluate(test_dataset)

d:\Dev\Hoang\AI\MathClassification\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Acc Label 0,Acc Label 1,Acc Label 2,Acc Label 3,Jaccard Samples,F1 Micro,F1 Macro,Precision Micro,Recall Micro
No log,0.169659,0,0.927464,0.935859,0.957167,0.925527,0.875305,0.883995,0.872662,0.901544,0.867117


{'eval_loss': 0.16965937614440918,
 'eval_acc_label_0': 0.9274644855789926,
 'eval_acc_label_1': 0.9358588032716315,
 'eval_acc_label_2': 0.9571674558760224,
 'eval_acc_label_3': 0.9255273353422299,
 'eval_jaccard_samples': 0.8753049217965273,
 'eval_f1_micro': 0.8839952811639795,
 'eval_f1_macro': 0.8726624285534688,
 'eval_precision_micro': 0.9015440144375376,
 'eval_recall_micro': 0.8671166827386693}